In [3]:
import pandas as pd
import re
import numpy as np
from collections import defaultdict

df = pd.read_csv('./blhs_2025_from_text.csv', encoding='utf-8-sig')

# Chỉ fillna cho các cột dạng object (văn bản)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('')
# Các cột số (float64, int64) không fillna hoặc có thể fill bằng 0 nếu muốn
# df.fillna(0, inplace=True)  # nếu cần

# Kiểm tra xem còn NaN nào không
print(df.isnull().sum())

Phần               0
Chương             0
Số điều            0
Tiêu đề điều       0
Nội dung điều      0
Số khoản          45
Nội dung khoản     0
Điểm               0
Nội dung điểm      0
dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_24588\4110320048.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [21]:
def combine_content(row):
    text = row['Nội dung điều']
    if row['Số khoản']:
        text += " " + row['Nội dung khoản']
    if row['Điểm']:
        text += " " + row['Nội dung điểm']
    return text.strip()

df['full_text'] = df.apply(combine_content, axis=1)

In [22]:
def extract_penalty_main(text):
    """Trả về danh sách hình phạt chính tìm thấy"""
    penalties = []
    # Các mẫu cơ bản
    patterns = [
        (r'phạt tiền', 'Phạt tiền'),
        (r'cải tạo không giam giữ', 'Cải tạo không giam giữ'),
        (r'phạt tù từ (\d+) năm đến (\d+) năm', lambda m: f"Phạt tù từ {m.group(1)} năm đến {m.group(2)} năm"),
        (r'phạt tù đến (\d+) năm', lambda m: f"Phạt tù đến {m.group(1)} năm"),
        (r'phạt tù từ (\d+) tháng đến (\d+) năm', lambda m: f"Phạt tù từ {m.group(1)} tháng đến {m.group(2)} năm"),
        (r'tù chung thân', 'Tù chung thân'),
        (r'tử hình', 'Tử hình')
    ]
    for pat, val in patterns:
        if isinstance(val, str):
            if re.search(pat, text, re.IGNORECASE):
                penalties.append(val)
        else:
            m = re.search(pat, text, re.IGNORECASE)
            if m:
                penalties.append(val(m))
    # Loại bỏ trùng
    return list(set(penalties)) if penalties else ['Không xác định']

In [23]:
def extract_penalty_supplementary(text):
    """Tìm hình phạt bổ sung"""
    supplements = []
    patterns = [
        (r'cấm đảm nhiệm chức vụ', 'Cấm đảm nhiệm chức vụ'),
        (r'cấm hành nghề', 'Cấm hành nghề'),
        (r'cấm cư trú', 'Cấm cư trú'),
        (r'quản chế', 'Quản chế'),
        (r'tước một số quyền công dân', 'Tước một số quyền công dân'),
        (r'tịch thu tài sản', 'Tịch thu tài sản'),
        (r'phạt tiền', 'Phạt tiền'),  # nếu là bổ sung
    ]
    # Ưu tiên các cụm "còn có thể bị" để xác định hình phạt bổ sung
    if re.search(r'còn có thể bị', text, re.IGNORECASE):
        for pat, val in patterns:
            if re.search(pat, text, re.IGNORECASE):
                supplements.append(val)
    return list(set(supplements)) if supplements else []

In [7]:
def get_crime_group(chapter_name):
    """Trả về nhóm tội từ tên chương (cột Chương)"""
    # Có thể mapping cứng hoặc lấy trực tiếp
    # Vì cột 'Chương' đã có dạng "Chương XIII - CÁC TỘI XÂM PHẠM AN NINH QUỐC GIA"
    # Ta lấy phần sau dấu '-'
    if '-' in chapter_name:
        group = chapter_name.split('-', 1)[1].strip()
    else:
        group = chapter_name.strip()
    return group

df['crime_group'] = df['Chương'].apply(get_crime_group)

In [8]:
def get_category_from_penalty(penalties):
    """Phân loại dựa trên hình phạt chính đầu tiên"""
    if not penalties or penalties[0] == 'Không xác định':
        return 'Không xác định'
    p = penalties[0].lower()
    if 'phạt tiền' in p or 'cải tạo không giam giữ' in p or 'tù đến 3 năm' in p or 'tù từ 3 tháng' in p:
        return 'Tội phạm ít nghiêm trọng'
    if 'tù từ 3 năm đến 7 năm' in p:
        return 'Tội phạm nghiêm trọng'
    if 'tù từ 7 năm đến 15 năm' in p:
        return 'Tội phạm rất nghiêm trọng'
    if 'tù trên 15 năm' in p or 'tù chung thân' in p or 'tử hình' in p:
        return 'Tội phạm đặc biệt nghiêm trọng'
    # Một số trường hợp tù có thời hạn nhưng không rõ
    if 'tù' in p:
        # Thử trích số năm tối đa
        m = re.search(r'(\d+) năm', p)
        if m:
            year = int(m.group(1))
            if year <= 3:
                return 'Tội phạm ít nghiêm trọng'
            elif year <= 7:
                return 'Tội phạm nghiêm trọng'
            elif year <= 15:
                return 'Tội phạm rất nghiêm trọng'
            else:
                return 'Tội phạm đặc biệt nghiêm trọng'
    return 'Không xác định'

# Áp dụng (tạm thời dùng penalty_main từ toàn bộ full_text)
df['temp_penalties'] = df['full_text'].apply(extract_penalty_main)
df['crime_category'] = df['temp_penalties'].apply(get_category_from_penalty)


In [9]:
def extract_actor(text):
    """Xác định chủ thể phạm tội"""
    if re.search(r'pháp nhân thương mại', text, re.IGNORECASE):
        return 'Pháp nhân thương mại'
    # Người có chức vụ?
    if re.search(r'lợi dụng chức vụ|người có chức vụ', text, re.IGNORECASE):
        return 'Người có chức vụ'
    return 'Người'

df['actor'] = df['full_text'].apply(extract_actor)

In [10]:
def extract_victim(text):
    """Trả về danh sách đối tượng bị hại"""
    victims = set()
    patterns = {
        'người khác': 'Người khác',
        'người dưới 16 tuổi': 'Người dưới 16 tuổi',
        'người dưới 18 tuổi': 'Người dưới 18 tuổi',
        'phụ nữ có thai': 'Phụ nữ có thai',
        'người già yếu': 'Người già yếu',
        'người đang thi hành công vụ': 'Người thi hành công vụ',
        'ông bà cha mẹ': 'Ông bà, cha mẹ',
        'đồng đội': 'Đồng đội',
        'người lệ thuộc': 'Người lệ thuộc',
        'người bị hại': 'Người bị hại'
    }
    for pattern, label in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            victims.add(label)
    # Nếu không có, mặc định là "Người khác"
    return list(victims) if victims else ['Người khác']

df['victim'] = df['full_text'].apply(extract_victim)

In [11]:
def extract_action(text):
    """Trích xuất hành vi chính từ văn bản"""
    actions = []
    # Danh sách từ khóa hành vi thường gặp (có thể mở rộng)
    action_keywords = [
        'giết người', 'cố ý gây thương tích', 'vô ý gây thương tích',
        'cướp tài sản', 'cướp giật', 'trộm cắp', 'lừa đảo',
        'buôn lậu', 'vận chuyển trái phép', 'tàng trữ trái phép',
        'sản xuất trái phép', 'mua bán trái phép',
        'tham ô', 'nhận hối lộ', 'đưa hối lộ',
        'làm giả', 'sửa chữa', 'khai báo gian dối',
        'dùng vũ lực', 'đe dọa', 'bắt cóc', 'chiếm đoạt',
        'hủy hoại tài sản', 'cố ý làm hư hỏng',
        'vi phạm an toàn giao thông', 'gây ô nhiễm môi trường',
        'trồng cây thuốc phiện', 'tổ chức đánh bạc', 'đánh bạc'
    ]
    for kw in action_keywords:
        if re.search(kw, text, re.IGNORECASE):
            actions.append(kw)
    # Nếu không tìm thấy, lấy động từ đầu câu?
    if not actions:
        # Lấy động từ đầu tiên (thử)
        sent = text.split('.')[0]
        # Có thể dùng underthesea để tách từ loại nhưng ở đây tạm bỏ qua
        pass
    return list(set(actions)) if actions else ['Không xác định']

df['action'] = df['full_text'].apply(extract_action)

In [12]:
def extract_consequence(text):
    consequences = []
    patterns = [
        (r'chết người', 'Chết người'),
        (r'tỷ lệ tổn thương cơ thể (\d+)%', 'Tỷ lệ thương tích {0}%'),
        (r'gây thương tích', 'Gây thương tích'),
        (r'thiệt hại về tài sản', 'Thiệt hại tài sản'),
        (r'gây rối loạn tâm thần', 'Rối loạn tâm thần'),
        (r'lây nhiễm HIV', 'Lây nhiễm HIV'),
        (r'gây ô nhiễm môi trường', 'Ô nhiễm môi trường'),
        (r'gây sự cố môi trường', 'Sự cố môi trường'),
        (r'gây ngộ độc', 'Ngộ độc'),
        (r'gây dịch bệnh', 'Dịch bệnh')
    ]
    for pat, label in patterns:
        if re.search(pat, text, re.IGNORECASE):
            # Thay thế nếu có nhóm
            m = re.search(pat, text, re.IGNORECASE)
            if m and '{0}' in label:
                label = label.format(m.group(1))
            consequences.append(label)
    return list(set(consequences)) if consequences else ['Không xác định']

df['consequence'] = df['full_text'].apply(extract_consequence)

In [13]:
aggravating_keywords = {
    'có tổ chức': 'Có tổ chức',
    'tính chất chuyên nghiệp': 'Tính chất chuyên nghiệp',
    'lợi dụng chức vụ, quyền hạn': 'Lợi dụng chức vụ',
    'tính chất côn đồ': 'Côn đồ',
    'động cơ đê hèn': 'Động cơ đê hèn',
    'cố tình thực hiện tội phạm đến cùng': 'Cố tình thực hiện đến cùng',
    'phạm tội 02 lần trở lên': 'Phạm tội nhiều lần',
    'tái phạm nguy hiểm': 'Tái phạm nguy hiểm',
    'đối với người dưới 16 tuổi': 'Phạm tội với người dưới 16 tuổi',
    'đối với phụ nữ có thai': 'Phạm tội với phụ nữ có thai',
    'đối với người già yếu': 'Phạm tội với người già yếu',
    'lợi dụng hoàn cảnh chiến tranh, thiên tai': 'Lợi dụng hoàn cảnh đặc biệt',
    'dùng thủ đoạn xảo quyệt': 'Thủ đoạn xảo quyệt',
    'xúi giục người dưới 18 tuổi': 'Xúi giục người dưới 18 tuổi',
}

def extract_aggravating(text):
    agg = []
    for key, label in aggravating_keywords.items():
        if re.search(key, text, re.IGNORECASE):
            agg.append(label)
    return list(set(agg)) if agg else []

df['aggravating_circ'] = df['full_text'].apply(extract_aggravating)


In [14]:
mitigating_keywords = {
    'ngăn chặn hoặc làm giảm bớt tác hại': 'Ngăn chặn tác hại',
    'tự nguyện sửa chữa, bồi thường thiệt hại': 'Tự nguyện sửa chữa, bồi thường',
    'phạm tội trong trường hợp vượt quá giới hạn phòng vệ chính đáng': 'Vượt quá phòng vệ chính đáng',
    'phạm tội trong trường hợp vượt quá yêu cầu của tình thế cấp thiết': 'Vượt quá tình thế cấp thiết',
    'phạm tội trong trường hợp bị kích động về tinh thần': 'Kích động tinh thần',
    'phạm tội vì hoàn cảnh đặc biệt khó khăn': 'Hoàn cảnh khó khăn',
    'phạm tội nhưng chưa gây thiệt hại': 'Chưa gây thiệt hại',
    'phạm tội lần đầu và thuộc trường hợp ít nghiêm trọng': 'Phạm tội lần đầu và ít nghiêm trọng',
    'phạm tội vì bị người khác đe dọa hoặc cưỡng bức': 'Bị đe dọa, cưỡng bức',
    'phạm tội trong trường hợp bị hạn chế khả năng nhận thức': 'Hạn chế nhận thức',
    'phạm tội do lạc hậu': 'Lạc hậu',
    'người phạm tội là phụ nữ có thai': 'Phụ nữ có thai',
    'người phạm tội là người đủ 70 tuổi trở lên': 'Người cao tuổi',
    'người phạm tội là người khuyết tật': 'Khuyết tật',
    'người phạm tội tự thú': 'Tự thú',
    'thành khẩn khai báo, ăn năn hối cải': 'Thành khẩn khai báo',
    'tích cực hợp tác với cơ quan có trách nhiệm': 'Tích cực hợp tác',
    'lập công chuộc tội': 'Lập công',
    'có thành tích xuất sắc': 'Thành tích xuất sắc',
    'người có công với cách mạng': 'Có công với cách mạng'
}

def extract_mitigating(text):
    mitig = []
    for key, label in mitigating_keywords.items():
        if re.search(key, text, re.IGNORECASE):
            mitig.append(label)
    return list(set(mitig)) if mitig else []

df['mitigating_circ'] = df['full_text'].apply(extract_mitigating)

In [15]:
def extract_condition(text):
    """Tìm các mệnh đề điều kiện"""
    conditions = []
    # Mẫu "nếu ..." hoặc "thuộc một trong các trường hợp"
    matches = re.findall(r'nếu\s+([^.]+)', text, re.IGNORECASE)
    conditions.extend(matches)
    matches = re.findall(r'thuộc một trong các trường hợp sau đây:\s*([^.]*)', text, re.IGNORECASE)
    conditions.extend(matches)
    # Giới hạn độ dài
    conditions = [c.strip()[:200] for c in conditions if c.strip()]
    return conditions if conditions else []

df['condition'] = df['full_text'].apply(extract_condition)

In [16]:
def extract_exception(text):
    """Tìm các câu bắt đầu bằng 'không áp dụng', 'trừ trường hợp'"""
    exceptions = []
    # Mẫu "không áp dụng ..."
    matches = re.findall(r'Không áp dụng\s+([^.]+)', text, re.IGNORECASE)
    exceptions.extend(matches)
    matches = re.findall(r'trừ trường hợp\s+([^.]+)', text, re.IGNORECASE)
    exceptions.extend(matches)
    exceptions = [e.strip()[:200] for e in exceptions if e.strip()]
    return exceptions if exceptions else []

df['exception'] = df['full_text'].apply(extract_exception)

In [17]:
def extract_judicial_measure(text):
    measures = []
    patterns = [
        (r'tịch thu vật, tiền liên quan đến tội phạm', 'Tịch thu vật, tiền'),
        (r'trả lại tài sản', 'Trả lại tài sản'),
        (r'bồi thường thiệt hại', 'Bồi thường thiệt hại'),
        (r'bắt buộc chữa bệnh', 'Bắt buộc chữa bệnh'),
        (r'khôi phục lại tình trạng ban đầu', 'Khôi phục tình trạng ban đầu'),
        (r'buộc công khai xin lỗi', 'Công khai xin lỗi')
    ]
    for pat, label in patterns:
        if re.search(pat, text, re.IGNORECASE):
            measures.append(label)
    return list(set(measures)) if measures else []

df['judicial_measure'] = df['full_text'].apply(extract_judicial_measure)

In [18]:
def extract_intent_type(text):
    if re.search(r'\bcố ý\b', text, re.IGNORECASE):
        if re.search(r'\bvô ý\b', text, re.IGNORECASE):
            return 'Cố ý hoặc vô ý'
        return 'Cố ý'
    elif re.search(r'\bvô ý\b', text, re.IGNORECASE):
        return 'Vô ý'
    else:
        return 'Không xác định'

df['intent_type'] = df['full_text'].apply(extract_intent_type)

In [19]:
# Đặt lại penalty_main từ full_text
df['penalty_main'] = df['full_text'].apply(extract_penalty_main)
df['penalty_supplementary'] = df['full_text'].apply(extract_penalty_supplementary)

# Chọn các cột cần lưu
output_columns = [
    'Phần', 'Chương', 'Số điều', 'Tiêu đề điều', 'Nội dung điều',
    'Số khoản', 'Nội dung khoản', 'Điểm', 'Nội dung điểm',
    'full_text',  # có thể giữ lại để đối chiếu
    'crime_group', 'crime_category', 'intent_type',
    'actor', 'victim', 'action', 'consequence',
    'penalty_main', 'penalty_supplementary',
    'aggravating_circ', 'mitigating_circ',
    'condition', 'exception', 'judicial_measure'
]

# Chuyển các cột list thành string để lưu CSV (dễ đọc)
for col in ['victim', 'action', 'consequence', 'penalty_main', 'penalty_supplementary',
            'aggravating_circ', 'mitigating_circ', 'condition', 'exception', 'judicial_measure']:
    df[col] = df[col].apply(lambda x: '; '.join(x) if isinstance(x, list) else x)

df_output = df[output_columns]

# Lưu ra file
df_output.to_csv('blhs_enriched.csv', index=False, encoding='utf-8-sig')
print("Đã lưu file blhs_enriched.csv thành công!")

Đã lưu file blhs_enriched.csv thành công!


In [20]:
sample = df_output.head(10)
sample

,Phần,Chương,Số điều,Tiêu đề điều,Nội dung điều,Số khoản,Nội dung khoản,Điểm,Nội dung điểm,full_text,...,victim,action,consequence,penalty_main,penalty_supplementary,aggravating_circ,mitigating_circ,condition,exception,judicial_measure
0,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,1.0,Nhiệm vụ của Bộ luật hình sự,Bộ luật hình sự có nhiệm vụ bảo vệ chủ quyền q...,NaN,,,,Bộ luật hình sự có nhiệm vụ bảo vệ chủ quyền q...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
1,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,2.0,Cơ sở của trách nhiệm hình sự,,1.0,Chỉ người nào phạm một tội đã được Bộ luật hìn...,,,Chỉ người nào phạm một tội đã được Bộ luật hìn...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
2,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,2.0,Cơ sở của trách nhiệm hình sự,,2.0,Chỉ pháp nhân thương mại nào phạm một tội đã đ...,,,Chỉ pháp nhân thương mại nào phạm một tội đã đ...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
3,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,a,Mọi hành vi phạm tội do người thực hiện phải đ...,Đối với người phạm tội: Mọi hành vi phạm tội d...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
4,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,b,Mọi người phạm tội đều bình đẳng trước pháp lu...,Đối với người phạm tội: Mọi người phạm tội đều...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
5,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,c,"Nghiêm trị người chủ mưu, cầm đầu, chỉ huy, ng...",Đối với người phạm tội: Nghiêm trị người chủ m...,...,Người khác,Không xác định,Không xác định,Không xác định,,Tái phạm nguy hiểm; Lợi dụng chức vụ,,,,
6,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,d,Nghiêm trị người phạm tội dùng thủ đoạn xảo qu...,Đối với người phạm tội: Nghiêm trị người phạm ...,...,Người khác,sửa chữa,Không xác định,Không xác định,,Tính chất chuyên nghiệp; Thủ đoạn xảo quyệt; C...,Lập công; Tích cực hợp tác,,,Bồi thường thiệt hại
7,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,đ,Đối với người lần đầu phạm tội ít nghiêm trọng...,Đối với người phạm tội: Đối với người lần đầu ...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
8,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,e,Đối với người bị phạt tù thì buộc họ phải chấp...,Đối với người phạm tội: Đối với người bị phạt ...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,"họ có đủ điều kiện do Bộ luật này quy định, th...",,
9,Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG,Chương I - ĐIỀU KHOẢN CƠ BẢN,3.0,Nguyên tắc xử lý,,1.0,Đối với người phạm tội:,g,Người đã chấp hành xong hình phạt được tạo điề...,Đối với người phạm tội: Người đã chấp hành xon...,...,Người khác,Không xác định,Không xác định,Không xác định,,,,,,
